Customer Churn Prediction

Machine Learning Model

Objective:

Build a predictive machine learning model to identify customers who are likely to churn. The model will help the business proactively retain high-risk customers by enabling targeted retention strategies.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Customer churn dataset.xlsx to Customer churn dataset.xlsx


In [4]:
df = pd.read_excel("Customer churn dataset.xlsx")

In [5]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,numAdminTickets,numTechTickets,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.5,0,0,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,0,0,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,3,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,0,0,Yes


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Encode Target Variable

In [7]:
df['Churn'] = df['Churn'].map({'Yes':1,'No':0})

Encode Categorical Columns

In [9]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

encoder = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    if col != 'customerID':  # Exclude customerID from encoding as it's an identifier
        df[col] = encoder.fit_transform(df[col])

/tmp/ipykernel_3359/664488476.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(0, inplace=True)


Separate Features & Target

In [10]:
X = df.drop('Churn',axis=1)

y = df['Churn']

Train Test Split

In [11]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Feature Scaling

In [12]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

Logistic Regression

Business Question:

Can Logistic Regression accurately predict customer churn?

In [13]:
lr = LogisticRegression()

lr.fit(X_train,y_train)

prediction = lr.predict(X_test)

Accuracy

In [14]:
accuracy_score(y_test,prediction)

0.8608942512420156

Classification Report

In [15]:
print(classification_report(y_test,prediction))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91      1036
           1       0.74      0.74      0.74       373

    accuracy                           0.86      1409
   macro avg       0.82      0.82      0.82      1409
weighted avg       0.86      0.86      0.86      1409



Confusion Matrix

In [16]:
confusion_matrix(y_test,prediction)

array([[938,  98],
       [ 98, 275]])

Decision Tree

In [17]:
tree = DecisionTreeClassifier(random_state=42)

tree.fit(X_train,y_train)

tree_pred = tree.predict(X_test)

accuracy_score(y_test,tree_pred)

0.8105039034776437

Random Forest

In [18]:
forest = RandomForestClassifier(random_state=42)

forest.fit(X_train,y_train)

forest_pred = forest.predict(X_test)

accuracy_score(y_test,forest_pred)

0.8559261887863733

Compare Models

In [19]:
results = pd.DataFrame({

'Model':['Logistic Regression',
         'Decision Tree',
         'Random Forest'],

'Accuracy':[

accuracy_score(y_test,prediction),

accuracy_score(y_test,tree_pred),

accuracy_score(y_test,forest_pred)

]

})

results

,Model,Accuracy
0,Logistic Regression,0.860894
1,Decision Tree,0.810504
2,Random Forest,0.855926


Feature Importance

In [20]:
importance = pd.DataFrame({

'Feature':X.columns,

'Importance':forest.feature_importances_

})

importance.sort_values(
'Importance',
ascending=False
)

,Feature,Importance
21,numTechTickets,0.158770
5,tenure,0.127232
19,TotalCharges,0.126179
18,MonthlyCharges,0.112727
0,customerID,0.091074
15,Contract,0.067659
9,OnlineSecurity,0.038369
17,PaymentMethod,0.034041
12,TechSupport,0.027628
8,InternetService,0.027118
